In [2]:
!pip install langchain
!pip install langchain-community
!pip install langchain-text-splitters
!pip install faiss-cpu
!pip install sentence-transformers
!pip install transformers
!pip install accelerate


In [56]:
from google.colab import files

uploaded = files.upload()

Saving About AI.txt to About AI (1).txt
Saving RAG_BasicDemo.py to RAG_BasicDemo (1).py


In [62]:
import os

print(os.listdir())

['.config', 'About AI.txt', '.ipynb_checkpoints', 'RAG_BasicDemo.py', 'RAG_BasicDemo (1).py', 'sample_data']


In [63]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("About AI.txt")

docs = loader.load()

print(docs)

[Document(metadata={'source': 'About AI.txt'}, page_content='Sri Ramakrishna Institute of Technology (SRIT) is located in Coimbatore.\n\nThe Principal of SRIT is Dr. S. Kumaravel.\n\nThe Head of the Computer Science and Engineering Department is Dr. R. Anitha.\n\nThe college library is open from 8:00 AM to 8:00 PM.\n\nMithun is a final-year B.E. Computer Science student.\n\nMithun is interested in Java, Artificial Intelligence, Full Stack Development, and Retrieval-Augmented Generation (RAG).')]


In [64]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print(len(chunks))

2


In [37]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:01<?, ?it/s]

In [65]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(chunks, embeddings)

retriever = db.as_retriever()

In [66]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_new_tokens=256,
    temperature=0.2
)

llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [67]:
def rag_chain(question):

    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are an AI assistant.

Use ONLY the context.

If you don't know, say I don't know.

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)

    return response.replace(prompt, "").strip()

In [69]:
print(rag_chain("What is Artificial Intelligence?"))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Artificial Intelligence refers to the simulation of human intelligence processes by computer systems, including learning, reasoning, and self-correction. It involves creating machines that can perform tasks requiring human-like cognitive abilities such as perception, understanding language, problem-solving, decision-making, and creativity. The field encompasses various subfields like machine learning, natural language processing, robotics, expert systems, and more. While Mithun's interest in Artificial Intelligence suggests he might be exploring this area within his studies or possibly pursuing it further, the definition provided here gives a broad overview of what artificial intelligence entails. 

Note: This answer provides a general explanation of Artificial Intelligence based on common knowledge about the topic. If specific details about Mithun's interests or the nature of his education were known, they could provide additional insights into how he might be applying or studying AI 

In [68]:
print(rag_chain("Who is the Principal?"))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dr. S. Kumaravel

Explanation:
Based on the given information, Sri Ramakrishna Institute of Technology (SRIT), which is located in Coimbatore, has a Principal named Dr. S. Kumaravel. This can be directly inferred from the statement "The Principal of SRIT is Dr. S. Kumaravel." Therefore, Dr. S. Kumaravel is the correct answer for who the Principal is. The other options provided do not match with the information given in the context.


In [70]:
print(rag_chain("Who is Mithun?"))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Mithun is a final-year B.E. Computer Science student at Sri Ramakrishna Institute of Technology (SRIT), which is located in Coimbatore. The principal of SRIT is Dr. S. Kumaravel, and the head of the Computer Science and Engineering Department is Dr. R. Anitha. The college library is open from 8:00 AM to 8:00 PM. Mithun's interests include Java, Artificial Intelligence, Full Stack Development, and Retrieval-Augmented Generation (RAG). However, without additional information, it cannot be determined if he has any specific projects or achievements related to these areas. Additionally, there is no mention of his participation in any competitions or awards. Therefore, based on the given context, we can only conclude that Mithun is a computer science student with some interests in technology-related fields. To provide more accurate details about him would require further information not provided in this context.
